In [1]:
import os
import json
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, '..')

from doc_extractor_utils import (
    parse_ar_summaries,
    get_recommended_summary_table_json,
    get_single_ar_summary_table,
    compare_ar_with_summary
)

from agents.summary_agent import (
    check_all_ar_summaries,
    analyze_with_llm,
    validate_ar_summary
)

print("✓ Imports successful!")
print(f"Python version: {sys.version}")

# Set your API key (if not already set)
# os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'


✓ Imports successful!
Python version: 3.10.18 (main, Jun  5 2025, 08:37:47) [Clang 14.0.6 ]


## 1. Load Extracted HTML Data


In [2]:
from document_extractor import extract_itac_report

doc_1_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report1/LS2502 - Final Draft R2.docx"
doc_2_path = "/Users/afschowdhury/Code Local/itac-report-validator/docs/report2/LS2508 - Final Draft.docx"
out = extract_itac_report(doc_2_path, output="html", save_files=True)

In [3]:
out.keys()

dict_keys(['general_information', 'annual_energy_usages_and_costs', 'carbon_footprint', 'recommendation_summary_table', 'ar_summary', 'assessment_recommendations'])

In [4]:
ar_summary_html = out['ar_summary']

rec_summary_html = out['recommendation_summary_table']

In [5]:


print("✓ Loaded HTML files")
print(f"  - AR summary HTML: {len(ar_summary_html)} characters")
print(f"  - Rec summary HTML: {len(rec_summary_html)} characters")


✓ Loaded HTML files
  - AR summary HTML: 3818 characters
  - Rec summary HTML: 7811 characters


## 2. Parse AR Summaries


In [6]:
# Parse AR summaries from HTML
ar_summaries = parse_ar_summaries(ar_summary_html)

print(f"✓ Found {len(ar_summaries)} AR summaries\n")

# Display first summary as example
if ar_summaries:
    first_ar = ar_summaries[0]
    print(f"Example - AR {first_ar['ar_no']}:")
    print(f"Summary: {first_ar['ar_summary'][:200]}...")
    print()

# Display all AR numbers found
ar_numbers = [ar['ar_no'] for ar in ar_summaries]
print(f"AR Numbers: {ar_numbers}")


✓ Found 8 AR summaries

Example - AR 1:
Summary: AR No. 1 – Utilize Higher Efficiency Lamps and/or Ballasts Utilizing higher-efficiency lamps and/or ballasts will enhance lighting system performance while reducing energy consumption. This energy eff...

AR Numbers: [1, 2, 3, 4, 5, 6, 7, 8]


In [36]:
ar_summary_1 = ar_summaries[0]
ar_summary_1

{'ar_no': 1,
 'ar_summary': 'AR No. 1 –\xa0Utilize Higher Efficiency Lamps and/or Ballasts Utilizing higher-efficiency lamps and/or ballasts will enhance lighting system performance while reducing energy consumption. This energy efficiency measure will yield $771 in total cost savings and incur an implementation cost of $675. The project has a payback period of 0.88 years and will reduce CO₂ emissions by 3 tons annually, with annual energy savings of 6,806 kWh.'}

## 3. Parse Recommendation Summary Table


In [7]:
# Parse recommendation summary table
rec_summary_data = get_recommended_summary_table_json(rec_summary_html)
recommendations = rec_summary_data.get('recommendations', [])

print(f"✓ Found {len(recommendations)} recommendations in summary table\n")

# Display first recommendation as example
if recommendations:
    first_rec = recommendations[0]
    print(f"Example - AR {first_rec.get('ar_number')}:")
    for key, value in first_rec.items():
        if key not in ['category', 'description']:
            print(f"  {key}: {value}")
    print()

print(f"Headers: {rec_summary_data.get('standardized_headers', [])}")


✓ Found 8 recommendations in summary table

Example - AR 1:
  ar_number: 1
  electricity_savings_kwh_per_year: 6806
  energy_cost_savings_per_year: 694
  demand_savings_kw_per_year: 17
  demandcost_savings_dollar_per_yr: 77
  admin_cost_savingsdollar_per_yr: 0
  propane_savingsmmbtu_per_yr: 0
  propanecost_savingdollar_per_yr: 0
  total_cost_savings_per_year: 771
  co2_reduction_tons_per_year: 3
  implementation_cost: 675
  payback_period_years: 0.88

Headers: ['ar_number', 'category', 'description', 'electricity_savings_kwh_per_year', 'energy_cost_savings_per_year', 'demand_savings_kw_per_year', 'demandcost_savings_dollar_per_yr', 'admin_cost_savingsdollar_per_yr', 'propane_savingsmmbtu_per_yr', 'propanecost_savingdollar_per_yr', 'total_cost_savings_per_year', 'co2_reduction_tons_per_year', 'implementation_cost', 'payback_period_years']


## 4. Load Individual AR Data


In [27]:

html_dir = Path('/Users/afschowdhury/Code Local/itac-report-validator/EXTRACTED_HTML')
ar_files = sorted(html_dir.glob("AR_*.html"))
ar_data_list = []

print(f"Loading {len(ar_files)} AR files...\n")

for ar_file in ar_files:
    with open(ar_file, 'r') as f:
        ar_html = f.read()
    
    ar_data = get_single_ar_summary_table(ar_html)
    
    if ar_data.get('ar_number'):
        ar_data_list.append(ar_data)
        print(f"✓ AR {ar_data['ar_number']}: {len(ar_data.get('data', {}))} data fields")

print(f"\n✓ Total ARs loaded: {len(ar_data_list)}")


Loading 8 AR files...

✓ AR 1: 8 data fields
✓ AR 2: 6 data fields
✓ AR 3: 7 data fields
✓ AR 4: 6 data fields
✓ AR 5: 6 data fields
✓ AR 6: 6 data fields
✓ AR 7: 6 data fields
✓ AR 8: 9 data fields

✓ Total ARs loaded: 8


In [34]:
ar_data_1 = ar_data_list[0]
ar_data_1['data']

{'electricity_savings_kwh_per_year': 6806,
 'energy_cost_savingsdollar_per_yr': 694,
 'demand_savings_kw_per_year': 17,
 'demand_costdollar_per_yr': 77,
 'total_cost_savings_per_year': 771,
 'co2reduction_tons_per_yr': 3,
 'implementation_cost': 675,
 'payback_period_years': 0.88}

In [37]:
ar_summary_1['ar_summary']

'AR No. 1 –\xa0Utilize Higher Efficiency Lamps and/or Ballasts Utilizing higher-efficiency lamps and/or ballasts will enhance lighting system performance while reducing energy consumption. This energy efficiency measure will yield $771 in total cost savings and incur an implementation cost of $675. The project has a payback period of 0.88 years and will reduce CO₂ emissions by 3 tons annually, with annual energy savings of 6,806 kWh.'

In [43]:
system_instruction = """
You are a helpful assistant that validates ITAC report summaries against numerical data.
You will be given a summary of a recommendations and numerical data.
You will need to validate the summary against the numerical data.
"""

user_message = f"""
Please validate the following summary against the numerical data.
Summary: {ar_summary_1['ar_summary']}
Numerical Data: {ar_data_1['data']}
"""


In [46]:
print(user_message)


Please validate the following summary against the numerical data.
Summary: AR No. 1 – Utilize Higher Efficiency Lamps and/or Ballasts Utilizing higher-efficiency lamps and/or ballasts will enhance lighting system performance while reducing energy consumption. This energy efficiency measure will yield $771 in total cost savings and incur an implementation cost of $675. The project has a payback period of 0.88 years and will reduce CO₂ emissions by 3 tons annually, with annual energy savings of 6,806 kWh.
Numerical Data: {'electricity_savings_kwh_per_year': 6806, 'energy_cost_savingsdollar_per_yr': 694, 'demand_savings_kw_per_year': 17, 'demand_costdollar_per_yr': 77, 'total_cost_savings_per_year': 771, 'co2reduction_tons_per_yr': 3, 'implementation_cost': 675, 'payback_period_years': 0.88}



In [50]:
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

response = client.models.generate_content(
    model="gemini-2.5-flash",
config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.1),
    contents=user_message
)
print(response.text)

The summary accurately reflects all the numerical data provided.

*   **Total cost savings:** $771 (Matches `total_cost_savings_per_year`: 771)
*   **Implementation cost:** $675 (Matches `implementation_cost`: 675)
*   **Payback period:** 0.88 years (Matches `payback_period_years`: 0.88)
*   **CO₂ emissions reduction:** 3 tons annually (Matches `co2reduction_tons_per_yr`: 3)
*   **Annual energy savings:** 6,806 kWh (Matches `electricity_savings_kwh_per_year`: 6806)


In [28]:
# Run validation (this doesn't require API key, just compares data)
gemini_api_key = os.getenv('GEMINI_API_KEY')
validation_results = check_all_ar_summaries(
    ar_summaries,
    recommendations,
    ar_data_list,
    api_key=gemini_api_key
)

print(f"✓ Validated {len(validation_results)} AR summaries\n")
print("=" * 70)

# Display results
for result in validation_results:
    ar_num = result.get('ar_number')
    
    if result.get('status') == 'error':
        print(f"❌ AR {ar_num}: {result.get('message')}")
        continue
    
    validation = result.get('validation', {})
    comparison = result.get('comparison', {})
    
    has_diffs = validation.get('has_differences', False)
    total_matches = validation.get('total_matches', 0)
    total_diffs = validation.get('total_differences', 0)
    
    status_icon = "⚠️ " if has_diffs else "✅"
    print(f"\n{status_icon} AR {ar_num}:")
    print(f"   Matches: {total_matches}, Differences: {total_diffs}")
    
    if has_diffs:
        print(f"   Discrepancies:")
        for diff in comparison.get('differences', []):
            field = diff.get('field')
            ar_val = diff.get('ar_value')
            summary_val = diff.get('summary_value')
            difference = diff.get('difference')
            print(f"     - {field}:")
            print(f"       AR value: {ar_val}")
            print(f"       Summary value: {summary_val}")
            print(f"       Difference: {difference}")

print("\n" + "=" * 70)


✓ Validated 8 AR summaries


✅ AR 1:
   Matches: 5, Differences: 0

✅ AR 2:
   Matches: 4, Differences: 0

✅ AR 3:
   Matches: 6, Differences: 0

✅ AR 4:
   Matches: 6, Differences: 0

✅ AR 5:
   Matches: 2, Differences: 0

✅ AR 6:
   Matches: 6, Differences: 0

✅ AR 7:
   Matches: 6, Differences: 0

✅ AR 8:
   Matches: 4, Differences: 0



In [29]:
# Save results to JSON
output_file = "ar_summary_validation_results.json"

with open(output_file, 'w') as f:
    json.dump(validation_results, f, indent=2)

print(f"✓ Validation results saved to: {output_file}")

# Display summary statistics
total_ars = len(validation_results)
ars_with_issues = sum(1 for r in validation_results if r.get('validation', {}).get('has_differences', False))
ars_clean = total_ars - ars_with_issues

print(f"\nSummary:")
print(f"  Total ARs: {total_ars}")
print(f"  ARs with issues: {ars_with_issues}")
print(f"  ARs without issues: {ars_clean}")


✓ Validation results saved to: ar_summary_validation_results.json

Summary:
  Total ARs: 8
  ARs with issues: 0
  ARs without issues: 8


In [30]:

if gemini_api_key:
    print("🤖 Generating AI-powered analysis using Gemini...")
    
    analysis = analyze_with_llm(validation_results, api_key=gemini_api_key)
    
    print("\n" + "=" * 70)
    print("AI ANALYSIS REPORT")
    print("=" * 70)
    print()
    print(analysis)
    
    # Save analysis to file
    with open("ar_summary_analysis.txt", 'w') as f:
        f.write(analysis)
    
    print("\n✓ Analysis saved to: ar_summary_analysis.txt")
else:
    print("⚠️  GOOGLE_API_KEY not set. Skipping AI analysis.")
    print("   To enable, run: os.environ['GOOGLE_API_KEY'] = 'your-key'")


print("Note: Uncomment the code above to run AI analysis")


🤖 Generating AI-powered analysis using Gemini...

AI ANALYSIS REPORT

## AR Summary Validation Report

This report analyzes the provided AR validation results, identifies data inconsistencies, discusses common patterns, assesses severity, and provides recommendations for corrections.

**1. ARs with Data Inconsistencies:**

*   **AR 1:** `demand_savings_kw_per_year` is present in Numerical Data Matches but is missing in Summary Text.
*   **AR 2:** `electricity_savings_kwh_per_year` and `demand_savings_kw_per_year` are missing in Numerical Data Matches.
*   **AR 5:** `total_cost_savings_per_year` and `co2_reduction_tons_per_year` are missing in Numerical Data Matches.
*   **AR 8:** `electricity_savings_kwh_per_year` and `demand_savings_kw_per_year` are missing in Numerical Data Matches.
*   **AR 4:** `energy_cost_savings_per_year` is present in Numerical Data Matches, it should be matched against `total_cost_savings_per_year`.
*   **AR 6:** `energy_cost_savings_per_year` is present in Nu